# Week 5: Forecasting spending

Goal: two forecasts per expense category, served by `POST /forecast` and shown in the app:
- **End of month:** *"At this pace, Food & Drinks will reach about ₱8,200 by Oct 31."*
- **Next month:** *"November: about ₱7,900 on Food & Drinks."*

You'll build features with no look-ahead, split by **time**, and check whether gradient boosting beats a simple baseline.

Spec: `docs/superpowers/specs/2026-10-23-spending-forecast-design.md`. Generate the data first: `.venv\Scripts\python scripts\generate_fake_data.py --years 3`

## Setup
The 3-year file has growth (~5% a year), a December bump and busier weekends, so there is something to learn. The dates are treated as the phone's local dates.

`CATEGORIES` is fixed: a category's position is its `category_code`, and the service uses the same list. The feature lists are the exact columns the service builds.

In [1]:
import calendar
import json
from datetime import date
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor

CATEGORIES = ['Food & Drinks', 'Coffee & Snacks', 'Groceries', 'Transportation', 'Bills & Utilities',
              'Shopping', 'Healthcare', 'Entertainment', 'other']
EOM_FEATURES = ['category_code', 'day', 'days_left', 'weekend_days_left', 'spent_ratio',
                'last_month_ratio', 'avg6_ratio', 'last_month_share_by_day', 'month_of_year']
NEXT_FEATURES = ['category_code', 'last_month_ratio', 'avg6_ratio', 'month_of_year']

df = pd.read_csv('../data/fake_transactions_3y.csv')
exp = df[df['Type'] == 'expense'].copy()
exp['day'] = pd.to_datetime(exp['Date']).dt.tz_localize(None).dt.normalize()

# Use whole months only: the file starts and ends mid-month
FIRST_DAY = exp['day'].min() + pd.offsets.MonthBegin(1)
LAST_FULL_MONTH = exp['day'].max().to_period('M').to_timestamp() - pd.offsets.MonthBegin(1)
LAST_DAY = LAST_FULL_MONTH + pd.offsets.MonthEnd(0)
exp = exp[(exp['day'] >= FIRST_DAY) & (exp['day'] <= LAST_DAY)]
print(FIRST_DAY.date(), '→', LAST_DAY.date(), len(exp), 'expenses')

2023-11-01 → 2026-09-30 4164 expenses


---
## A. Daily table
Build three tables, one column per category:
- `daily`: total spent per **day**. Hint: `pivot_table(index='day', columns='Category', values='Amount', aggfunc='sum', fill_value=0)`, then `reindex(pd.date_range(FIRST_DAY, LAST_DAY, freq='D'), fill_value=0)` so days with no spending are 0
- `monthly`: total per month. Hint: `daily.resample('MS').sum()` (index = first day of each month)
- `cum`: running total **within** each month. Hint: `daily.groupby(daily.index.to_period('M')).cumsum()`

In [2]:
# TODO
# Daily total per category, with every day present (no spending = 0)
daily = exp.pivot_table(index='day', columns='Category', values='Amount',
                        aggfunc='sum', fill_value=0)
daily = daily.reindex(pd.date_range(FIRST_DAY, LAST_DAY, freq='D'), fill_value=0)
daily = daily.reindex(columns=CATEGORIES, fill_value=0)   # fixed order = category_code
daily.index.name = 'day'
daily.columns.name = None

# Monthly totals (index = first day of each month)
monthly = daily.resample('MS').sum()

# Running total within each month
cum = daily.groupby(daily.index.to_period('M')).cumsum()

# Quick checks
assert len(daily) == (LAST_DAY - FIRST_DAY).days + 1
assert np.allclose(cum[cum.index.is_month_end].values, monthly.values)
print(daily.shape, monthly.shape, cum.shape)
monthly.tail()

(1065, 9) (35, 9) (1065, 9)


,Food & Drinks,Coffee & Snacks,Groceries,Transportation,Bills & Utilities,Shopping,Healthcare,Entertainment,other
day,,,,,,,,,
2026-05-01,15517.68,3995.55,24044.61,15618.95,8260.59,10981.04,4187.49,1996.09,0
2026-06-01,11158.27,3197.21,20495.64,16920.39,8589.38,19770.87,5141.03,2980.55,0
2026-07-01,12832.93,3513.09,21680.50,19880.15,6912.64,17266.33,2685.24,7223.60,0
2026-08-01,16040.07,3098.74,16764.44,15213.63,6900.44,11230.06,2851.45,1630.00,0
2026-09-01,12522.14,2503.62,12230.07,20509.56,8097.94,37343.72,6516.51,3054.64,0


---
## B. Features (no look-ahead)
### B1. End-of-month rows
One row per **(category, month M, cut-off day d = 1…28)**. "Full months" are the months before M, starting at the category's first month with spending. Skip M if it has fewer than **6** full months before it, if `avg3 <= 0`, or if M is after `LAST_FULL_MONTH`.

| column | meaning |
|---|---|
| `category`, `month`, `spent`, `avg3`, `actual` | kept for scoring (`actual` = total of month M) |
| `category_code` | `CATEGORIES.index(category)` |
| `day`, `days_left` | d, and days in M minus d |
| `weekend_days_left` | Saturdays + Sundays from day d+1 to the end of M |
| `spent_ratio` | spent on days 1…d of M, ÷ `avg3` |
| `last_month_ratio`, `avg6_ratio` | last full month's total ÷ `avg3`; mean of the last 6 full months ÷ `avg3` |
| `last_month_share_by_day` | share of last month's total spent by the same day last month (use day `min(d, days in last month)`); 0 if last month was 0 |
| `month_of_year` | M's month number |
| `eom_ratio` (**target**) | (`actual` − `spent`) ÷ `avg3`, the rest of the month in "usual months" |

Hints: loop over `daily.columns` and `monthly.index`. `hist = tot[(tot.index >= first) & (tot.index < M)]` gives the full months. `cum.loc[M + pd.Timedelta(days=d - 1), cat]` is spent through day d. `M.days_in_month` and `date(y, m, k).weekday() >= 5` help with the calendar. Build a list of dicts, then `eom = pd.DataFrame(rows)`.

**Think:** why divide everything by `avg3` instead of predicting pesos? (Hint: will a real user spend like the fake one?)

In [3]:
# TODO
rows = []
for cat in daily.columns:
    tot = monthly[cat]
    if not (tot > 0).any():
        continue                              # category never used
    first = tot[tot > 0].index.min()          # first month with spending
    code = CATEGORIES.index(cat)

    for M in monthly.index:
        if M > LAST_FULL_MONTH:
            continue
        hist = tot[(tot.index >= first) & (tot.index < M)]   # full months before M
        if len(hist) < 6:
            continue
        avg3 = hist.iloc[-3:].mean()
        if avg3 <= 0:
            continue

        last = hist.index[-1]                 # last full month
        last_total = hist.iloc[-1]
        avg6 = hist.iloc[-6:].mean()
        dim = M.days_in_month
        dim_last = last.days_in_month
        actual = tot.loc[M]

        for d in range(1, 29):
            spent = cum.loc[M + pd.Timedelta(days=d - 1), cat]
            weekend_left = sum(date(M.year, M.month, k).weekday() >= 5
                               for k in range(d + 1, dim + 1))
            if last_total > 0:
                same_day = cum.loc[last + pd.Timedelta(days=min(d, dim_last) - 1), cat]
                share = same_day / last_total
            else:
                share = 0.0

            rows.append({
                'category': cat, 'month': M, 'spent': spent, 'avg3': avg3, 'actual': actual,
                'category_code': code,
                'day': d,
                'days_left': dim - d,
                'weekend_days_left': weekend_left,
                'spent_ratio': spent / avg3,
                'last_month_ratio': last_total / avg3,
                'avg6_ratio': avg6 / avg3,
                'last_month_share_by_day': share,
                'month_of_year': M.month,
                'eom_ratio': (actual - spent) / avg3,
            })

eom = pd.DataFrame(rows)

# Checks
assert eom[EOM_FEATURES].notna().all().all()
assert (eom['spent'] + eom['eom_ratio'] * eom['avg3'] - eom['actual']).abs().max() < 1e-6
print(eom.shape, eom['month'].min().date(), '→', eom['month'].max().date())
eom.head()

(6496, 15) 2024-05-01 → 2026-09-01


,category,month,spent,avg3,actual,category_code,day,days_left,weekend_days_left,spent_ratio,last_month_ratio,avg6_ratio,last_month_share_by_day,month_of_year,eom_ratio
0,Food & Drinks,2024-05-01,770.16,12179.506667,14228.19,0,1,30,8,0.063234,1.052596,1.088269,0.009817,5,1.104973
1,Food & Drinks,2024-05-01,1201.41,12179.506667,14228.19,0,2,29,8,0.098642,1.052596,1.088269,0.017015,5,1.069565
2,Food & Drinks,2024-05-01,1741.56,12179.506667,14228.19,0,3,28,8,0.142991,1.052596,1.088269,0.154471,5,1.025216
3,Food & Drinks,2024-05-01,2482.01,12179.506667,14228.19,0,4,27,7,0.203786,1.052596,1.088269,0.162131,5,0.964422
4,Food & Drinks,2024-05-01,3444.21,12179.506667,14228.19,0,5,26,6,0.282787,1.052596,1.088269,0.171045,5,0.885420


### B2. Next-month rows
One row per **(category, month M)**: "we are somewhere in M, predict M+1". Only **full months before M** are used, since M itself is unfinished.

Columns: `category`, `month` (= M), `avg3`, `actual` (= total of M+1), `category_code`, `last_month_ratio`, `avg6_ratio`, `month_of_year` (of **M+1**), and the target `next_ratio` = total(M+1) ÷ `avg3`. Same skip rules; also skip when M+1 is after `LAST_FULL_MONTH`. Save as `nxt`.

In [4]:
# TODO
rows = []
for cat in daily.columns:
    tot = monthly[cat]
    if not (tot > 0).any():
        continue
    first = tot[tot > 0].index.min()
    code = CATEGORIES.index(cat)

    for M in monthly.index:
        nextM = M + pd.offsets.MonthBegin(1)
        if nextM > LAST_FULL_MONTH:
            continue
        hist = tot[(tot.index >= first) & (tot.index < M)]   # full months before M
        if len(hist) < 6:
            continue
        avg3 = hist.iloc[-3:].mean()
        if avg3 <= 0:
            continue

        actual = tot.loc[nextM]
        rows.append({
            'category': cat, 'month': M, 'avg3': avg3, 'actual': actual,
            'category_code': code,
            'last_month_ratio': hist.iloc[-1] / avg3,
            'avg6_ratio': hist.iloc[-6:].mean() / avg3,
            'month_of_year': nextM.month,
            'next_ratio': actual / avg3,
        })

nxt = pd.DataFrame(rows)

# Checks
assert nxt[NEXT_FEATURES].notna().all().all()
assert ((nxt['next_ratio'] * nxt['avg3'] - nxt['actual']).abs() < 1e-6).all()
assert (nxt['month'] + pd.offsets.MonthBegin(1) <= LAST_FULL_MONTH).all()
print(nxt.shape, nxt['month'].min().date(), '→', nxt['month'].max().date())
nxt.head()

(224, 9) 2024-05-01 → 2026-08-01


,category,month,avg3,actual,category_code,last_month_ratio,avg6_ratio,month_of_year,next_ratio
0,Food & Drinks,2024-05-01,12179.506667,14476.39,0,1.052596,1.088269,6,1.188586
1,Food & Drinks,2024-06-01,13964.286667,14844.37,0,1.018898,0.974619,7,1.063024
2,Food & Drinks,2024-07-01,13841.560000,14389.83,0,1.045864,0.928083,8,1.039610
3,Food & Drinks,2024-08-01,14516.316667,12617.55,0,1.022599,0.919511,9,0.869198
4,Food & Drinks,2024-09-01,14570.196667,15055.49,0,0.987621,0.979207,10,1.033307


These checks recompute one row of each table straight from the raw expenses. If they pass, there is no look-ahead.

In [5]:
def month_total(cat, start):
    m = exp[(exp['Category'] == cat) & (exp['day'] >= start) & (exp['day'] < start + pd.offsets.MonthBegin(1))]
    return m['Amount'].sum()

r = eom[(eom['day'] == 10)].iloc[len(eom[eom['day'] == 10]) // 2]
past = [month_total(r['category'], r['month'] - pd.offsets.MonthBegin(k)) for k in (1, 2, 3)]
spent = exp[(exp['Category'] == r['category']) & (exp['day'] >= r['month']) & (exp['day'] <= r['month'] + pd.Timedelta(days=9))]['Amount'].sum()
assert abs(r['avg3'] - np.mean(past)) < 1e-6 and abs(r['spent_ratio'] - spent / np.mean(past)) < 1e-9
assert abs(r['eom_ratio'] - (month_total(r['category'], r['month']) - spent) / np.mean(past)) < 1e-9

n = nxt.iloc[len(nxt) // 2]
past = [month_total(n['category'], n['month'] - pd.offsets.MonthBegin(k)) for k in (1, 2, 3)]
assert abs(n['last_month_ratio'] - past[0] / np.mean(past)) < 1e-9
assert abs(n['next_ratio'] - month_total(n['category'], n['month'] + pd.offsets.MonthBegin(1)) / np.mean(past)) < 1e-9
assert list(eom[EOM_FEATURES].columns) == EOM_FEATURES and list(nxt[NEXT_FEATURES].columns) == NEXT_FEATURES
print('no look-ahead ✓')

no look-ahead ✓


---
## C. Split by time
Test = the **last 6 full months**, train = everything before. For `nxt`, a row belongs to the month it **predicts** (M+1).

Make `eom_train`, `eom_test`, `nxt_train`, `nxt_test`. Hint: `test_from = LAST_FULL_MONTH - pd.offsets.MonthBegin(5)`.

**Think:** why would `train_test_split(..., shuffle=True)` give a score that's too good? (Hint: rows from the same month share almost everything.)

In [6]:
# TODO
test_from = LAST_FULL_MONTH - pd.offsets.MonthBegin(5)     # first of the last 6 full months

# End of month: a row belongs to its month M
eom_train = eom[eom['month'] <  test_from]
eom_test  = eom[eom['month'] >= test_from]

# Next month: a row belongs to the month it predicts, M+1
nxt_target = nxt['month'] + pd.offsets.MonthBegin(1)
nxt_train = nxt[nxt_target <  test_from]
nxt_test  = nxt[nxt_target >= test_from]

# Checks: no target month appears on both sides, and test is exactly 6 months
assert eom_train['month'].max() < eom_test['month'].min()
assert (nxt_train['month'] + pd.offsets.MonthBegin(1)).max() < test_from
assert eom_test['month'].nunique() == 6
assert (nxt_test['month'] + pd.offsets.MonthBegin(1)).nunique() == 6

print('test from', test_from.date(), '→', LAST_FULL_MONTH.date())
print(f'eom: {len(eom_train)} train / {len(eom_test)} test')
print(f'nxt: {len(nxt_train)} train / {len(nxt_test)} test')

test from 2026-04-01 → 2026-09-01
eom: 5152 train / 1344 test
nxt: 176 train / 48 test


---
## D. End-of-month model
Scoring helper (given): turns a predicted ratio back into pesos and returns the mean absolute error.

In [8]:
def pesos(pred_ratio, rows):
    """Predicted month total in pesos (end of month) or next month's total."""
    spent = rows['spent'] if 'spent' in rows else 0
    return spent + np.clip(pred_ratio, 0, None) * rows['avg3']


def mae_pesos(pred_ratio, rows):
    return float((pesos(pred_ratio, rows) - rows['actual']).abs().mean())

Compare three predictions of `eom_ratio` on `eom_test`, and save them in a dict `eom_preds`:
- **baseline:** the pace so far: `spent / day × days_in_month`, turned into a ratio: `(that − spent) / avg3`
- **HistGB:** `HistGradientBoostingRegressor(random_state=0)` fitted on `eom_train[EOM_FEATURES]` → `eom_ratio`. Save the model as `hgb_eom`
- **LightGBM:** `lgb.LGBMRegressor(random_state=0, verbose=-1)`, same data. Save as `lgb_eom`

Print `mae_pesos(...)` for each.

In [9]:
# TODO
# Baseline: keep spending at the same daily pace for the rest of the month
dim = eom_test['month'].dt.days_in_month
pace_total = eom_test['spent'] / eom_test['day'] * dim
baseline = (pace_total - eom_test['spent']) / eom_test['avg3']

# HistGradientBoosting
hgb_eom = HistGradientBoostingRegressor(random_state=0)
hgb_eom.fit(eom_train[EOM_FEATURES], eom_train['eom_ratio'])

# LightGBM
lgb_eom = lgb.LGBMRegressor(random_state=0, verbose=-1)
lgb_eom.fit(eom_train[EOM_FEATURES], eom_train['eom_ratio'])

eom_preds = {
    'baseline': baseline.to_numpy(),
    'HistGB':   hgb_eom.predict(eom_test[EOM_FEATURES]),
    'LightGBM': lgb_eom.predict(eom_test[EOM_FEATURES]),
}

for name, pred in eom_preds.items():
    print(f'{name:9s} MAE ₱{mae_pesos(pred, eom_test):,.0f}')

baseline  MAE ₱4,598
HistGB    MAE ₱2,751
LightGBM  MAE ₱2,727


### D2. Error by cut-off day
Show each method's MAE grouped by `day` (rows 1, 3, 5, 7, 10, 15, 20, 25, 28). Hint: `(pesos(pred, eom_test) - eom_test['actual']).abs().groupby(eom_test['day']).mean()`.

**Think:** Ollie warns "on pace to go over budget" from **day 5**. Is the error at day 5 small enough, or should it wait longer? And which method wins near the **end** of the month? Why might that be?

In [10]:
# TODO
days = [1, 3, 5, 7, 10, 15, 20, 25, 28]

by_day = pd.DataFrame({
    name: (pesos(pred, eom_test) - eom_test['actual']).abs()
              .groupby(eom_test['day']).mean()
    for name, pred in eom_preds.items()
}).loc[days].round(0)

# Same errors as a % of a usual month, so "is it small enough?" has a scale
usual = eom_test.groupby('day')['avg3'].mean().loc[days]
pct = by_day.div(usual, axis=0).mul(100).round(1).add_suffix(' %')

pd.concat([by_day, pct], axis=1)

,baseline,HistGB,LightGBM,baseline %,HistGB %,LightGBM %
day,,,,,,
1,9633.0,3908.0,3953.0,87.6,35.6,36.0
3,7496.0,3866.0,3793.0,68.2,35.2,34.5
5,10207.0,3813.0,3800.0,92.9,34.7,34.6
7,7513.0,3601.0,3550.0,68.4,32.8,32.3
10,5719.0,3251.0,3151.0,52.0,29.6,28.7
15,4159.0,2614.0,2686.0,37.8,23.8,24.4
20,2285.0,2109.0,2104.0,20.8,19.2,19.1
25,1493.0,1746.0,1730.0,13.6,15.9,15.7
28,753.0,1181.0,1095.0,6.9,10.7,10.0


---
## E. Next-month model
Same comparison on `nxt_test`, saved as `next_preds`:
- **baseline:** `avg3`, which is a ratio of exactly 1
- **HistGB** (`hgb_next`) and **LightGBM** (`lgb_next`; try `min_child_samples=5`, there are few rows) on `NEXT_FEATURES` → `next_ratio`

**Think:** how many training rows does this model have, compared with D? What does that do to boosting?

In [11]:
# TODO
# Baseline: next month = usual month (ratio 1)
baseline_next = np.ones(len(nxt_test))

# HistGradientBoosting
hgb_next = HistGradientBoostingRegressor(random_state=0)
hgb_next.fit(nxt_train[NEXT_FEATURES], nxt_train['next_ratio'])

# LightGBM (few rows, so allow small leaves)
lgb_next = lgb.LGBMRegressor(random_state=0, verbose=-1, min_child_samples=5)
lgb_next.fit(nxt_train[NEXT_FEATURES], nxt_train['next_ratio'])

next_preds = {
    'baseline': baseline_next,
    'HistGB':   hgb_next.predict(nxt_test[NEXT_FEATURES]),
    'LightGBM': lgb_next.predict(nxt_test[NEXT_FEATURES]),
}

print(f'train rows: nxt {len(nxt_train)} vs eom {len(eom_train)}')
for name, pred in next_preds.items():
    print(f'{name:9s} MAE ₱{mae_pesos(pred, nxt_test):,.0f}')

train rows: nxt 176 vs eom 5152
baseline  MAE ₱2,994
HistGB    MAE ₱3,772
LightGBM  MAE ₱4,203


---
## F. Decide
**Gate:** a model ships only if its test MAE is **lower than the baseline's**. Otherwise the baseline ships.

Set `EOM_CHOICE` and `NEXT_CHOICE` to `'HistGB'`, `'LightGBM'` or `'baseline'`, and put your fitted models in `FITTED` (`{'eom': {'HistGB': hgb_eom, 'LightGBM': lgb_eom}, 'next': {...}}`). Then run the cells below.

LightGBM may not load on Vercel. If it wins by only a little, HistGB is the safer pick.

In [12]:
# TODO
FITTED = {
    'eom':  {'HistGB': hgb_eom,  'LightGBM': lgb_eom},
    'next': {'HistGB': hgb_next, 'LightGBM': lgb_next},
}

def choose(preds, rows, margin=0.03):
    """Gate: a model ships only if it beats the baseline.
    Prefer HistGB unless LightGBM is better by more than `margin` (3%)."""
    mae = {name: mae_pesos(p, rows) for name, p in preds.items()}
    models = {k: v for k, v in mae.items() if k != 'baseline' and v < mae['baseline']}
    if not models:
        pick = 'baseline'
    elif 'HistGB' in models and 'LightGBM' in models:
        pick = 'LightGBM' if models['LightGBM'] < models['HistGB'] * (1 - margin) else 'HistGB'
    else:
        pick = min(models, key=models.get)
    for k, v in mae.items():
        print(f'  {k:9s} ₱{v:,.0f}' + ('  ← ship' if k == pick else ''))
    return pick

print('End of month:'); EOM_CHOICE  = choose(eom_preds,  eom_test)
print('Next month:');   NEXT_CHOICE = choose(next_preds, nxt_test)
print(EOM_CHOICE, NEXT_CHOICE)


End of month:
  baseline  ₱4,598
  HistGB    ₱2,751  ← ship
  LightGBM  ₱2,727
Next month:
  baseline  ₱2,994  ← ship
  HistGB    ₱3,772
  LightGBM  ₱4,203
HistGB baseline


In [13]:
# Gate (given)
for name, preds, choice in [('end of month', eom_preds, EOM_CHOICE), ('next month', next_preds, NEXT_CHOICE)]:
    scores = {k: mae_pesos(v, eom_test if name == 'end of month' else nxt_test) for k, v in preds.items()}
    ok = choice == 'baseline' or scores[choice] < scores['baseline']
    print(f"{name}: ships {choice} (MAE ₱{scores[choice]:,.0f} vs baseline ₱{scores['baseline']:,.0f})", '✓' if ok else '✗ fails the gate')
    assert ok

end of month: ships HistGB (MAE ₱2,751 vs baseline ₱4,598) ✓
next month: ships baseline (MAE ₱2,994 vs baseline ₱2,994) ✓


In [14]:
# Export (given): models for the service, plus a sample the service test compares its features against
def bundle(choice, kind, train, features, target):
    if choice == 'baseline':
        return {'model': None, 'features': features}
    model = clone(FITTED[kind][choice]).fit(train[features].to_numpy(), train[target].to_numpy())  # NumPy in, so the service needs no pandas
    return {'model': model, 'features': features}

service = Path('../service')
joblib.dump(bundle(EOM_CHOICE, 'eom', eom_train, EOM_FEATURES, 'eom_ratio'), service / 'model' / 'forecast_eom.joblib')
joblib.dump(bundle(NEXT_CHOICE, 'next', nxt_train, NEXT_FEATURES, 'next_ratio'), service / 'model' / 'forecast_next.joblib')

TODAY = pd.Timestamp('2026-08-15')
month = TODAY.to_period('M').to_timestamp()
start = month - pd.offsets.MonthBegin(6)
sample = {'request': {'today': str(TODAY.date()), 'series': []}, 'eom': {}, 'next': {}}
for key, cat in [('food', 'Food & Drinks'), ('bills', 'Bills & Utilities')]:
    days = daily.loc[start:TODAY, cat]
    sample['request']['series'].append({'key': key, 'category': cat,
                                        'daily': {str(d.date()): round(float(v), 2) for d, v in days.items() if v > 0}})
    e = eom[(eom['category'] == cat) & (eom['month'] == month) & (eom['day'] == TODAY.day)]
    n = nxt[(nxt['category'] == cat) & (nxt['month'] == month)]
    sample['eom'][key] = {k: float(v) for k, v in e[EOM_FEATURES].iloc[0].items()}
    sample['next'][key] = {k: float(v) for k, v in n[NEXT_FEATURES].iloc[0].items()}
(service / 'tests' / 'fixtures').mkdir(exist_ok=True)
(service / 'tests' / 'fixtures' / 'forecast_sample.json').write_text(json.dumps(sample, indent=1))
print('saved models and sample ✓')

saved models and sample ✓


**Note:** the sample uses amounts rounded to centavos, the same as the app sends, so its features can differ from the table in the 10th decimal. The service test allows for that.

**My choices:** TODO (end of month: …, next month: …)

**Why:** TODO, with the MAE numbers. Also: is day 5 safe for Ollie's pace warning? (D2)

My choices: end of month: 'HistGB', next month: 'baseline'

Why: 
- End of month: baseline ₱4,598, HistGB ₱2,751, LightGBM ₱2,727. Both models beat the pace baseline by about 40%. LightGBM is only 0.9% better than HistGB (₱24), well under the 3% margin, so HistGB ships: it's part of scikit-learn and loads on Vercel without LightGBM's native library.
- Next month: baseline ₱2,994, HistGB ₱3,772, LightGBM ₱4,203. Neither model beats "next month ≈ avg3": they're 26% and 40% worse. With about 28× fewer training rows than the end-of-month model, boosting fits noise from individual months, so the baseline ships.

In [15]:
print(pd.concat([by_day['HistGB'], pct['HistGB %']], axis=1).to_string())

     HistGB  HistGB %
day                  
1    3908.0      35.6
3    3866.0      35.2
5    3813.0      34.7
7    3601.0      32.8
10   3251.0      29.6
15   2614.0      23.8
20   2109.0      19.2
25   1746.0      15.9
28   1181.0      10.7


**Day 5 pace warning:** not safe. At day 5 HistGB is off by ₱3,813 on average, about 35% of a usual month, so a forecast 20–30% over budget is within normal error and would often be a false alarm. The error falls slowly (33% at day 7, 30% at day 10, 24% at day 15) and only gets under 20% at day 20 (₱2,109, 19%). Ollie should start the pace warning at day 20, or, if it warns earlier, only when the forecast is over budget by more than that day's typical error (more than 35% over at day 5, 24% at day 15).